In [1]:
import pandas as pd
import numpy as np

# Load the cleaned dataset from Day 3
df = pd.read_csv("../data/processed/orders_cleaned.csv")
print(f"Loaded cleaned data: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Loaded cleaned data: 39141 rows, 26 columns
Columns: ['order_id', 'courier_id', 'courier_age', 'courier_rating', 'restaurant_lat', 'restaurant_lng', 'delivery_lat', 'delivery_lng', 'order_date', 'time_ordered', 'time_picked', 'weather', 'traffic', 'vehicle_condition', 'order_type', 'vehicle_type', 'multi_deliveries', 'is_festival', 'city_type', 'delivery_minutes', 'distance_km', 'order_hour', 'day_of_week', 'is_weekend', 'is_peak_hour', 'is_on_time']


In [2]:
# ==============================================================
# DIM_COURIER — one row per unique courier
# ==============================================================
# A courier has stable attributes (age, rating, vehicle condition)
# that don't change per order. So we deduplicate.

dim_courier = df[['courier_id', 'courier_age', 'courier_rating', 
                   'vehicle_condition']].drop_duplicates(subset='courier_id').reset_index(drop=True)

print(f"dim_courier: {dim_courier.shape[0]} unique couriers")
print(dim_courier.head())

dim_courier: 1170 unique couriers
        courier_id  courier_age  courier_rating  vehicle_condition
0   INDORES13DEL02           37             4.9                  2
1   BANGRES18DEL02           34             4.5                  2
2   BANGRES19DEL01           23             4.4                  0
3  COIMBRES13DEL02           38             4.7                  0
4   CHENRES12DEL01           32             4.6                  1


In [3]:
# ==============================================================
# DIM_RESTAURANT — one row per unique restaurant location
# ==============================================================
# The dataset doesn't have a restaurant_id. We create one from unique 
# lat/lng combos, since a unique location = unique restaurant.

# Create a restaurant_id based on lat/lng combination
df['restaurant_key'] = (df['restaurant_lat'].round(4).astype(str) + '_' + 
                        df['restaurant_lng'].round(4).astype(str))

dim_restaurant = df[['restaurant_key', 'restaurant_lat', 
                     'restaurant_lng']].drop_duplicates(subset='restaurant_key').reset_index(drop=True)

# Create a clean surrogate key (integer)
dim_restaurant['restaurant_id'] = range(1, len(dim_restaurant) + 1)
dim_restaurant = dim_restaurant[['restaurant_id', 'restaurant_key', 
                                  'restaurant_lat', 'restaurant_lng']]

print(f"dim_restaurant: {dim_restaurant.shape[0]} unique restaurants")
print(dim_restaurant.head())

dim_restaurant: 388 unique restaurants
   restaurant_id   restaurant_key  restaurant_lat  restaurant_lng
0              1   22.745_75.8925       22.745049       75.892471
1              2   12.913_77.6832       12.913041       77.683237
2              3  12.9143_77.6784       12.914264       77.678400
3              4  11.0037_76.9765       11.003669       76.976494
4              5    12.9728_80.25       12.972793       80.249982


In [4]:
# ==============================================================
# DIM_DATE — one row per unique date
# ==============================================================
# Dates have stable attributes (day of week, weekend flag, month, year)
# Separating them lets us filter "all weekends" or "all January" easily.

df['order_date'] = pd.to_datetime(df['order_date'])

dim_date = df[['order_date']].drop_duplicates().reset_index(drop=True)
dim_date['date_id'] = range(1, len(dim_date) + 1)
dim_date['day_of_week'] = dim_date['order_date'].dt.day_name()
dim_date['is_weekend'] = dim_date['order_date'].dt.dayofweek >= 5
dim_date['month'] = dim_date['order_date'].dt.month_name()
dim_date['year'] = dim_date['order_date'].dt.year

dim_date = dim_date[['date_id', 'order_date', 'day_of_week', 
                      'is_weekend', 'month', 'year']]

print(f"dim_date: {dim_date.shape[0]} unique dates")
print(dim_date.head())

dim_date: 44 unique dates
   date_id order_date day_of_week  is_weekend  month  year
0        1 2022-03-19    Saturday        True  March  2022
1        2 2022-03-25      Friday       False  March  2022
2        3 2022-04-05     Tuesday       False  April  2022
3        4 2022-03-26    Saturday        True  March  2022
4        5 2022-03-11      Friday       False  March  2022


In [5]:
# ==============================================================
# DIM_LOCATION — one row per unique delivery city type
# ==============================================================

dim_location = df[['city_type']].drop_duplicates().reset_index(drop=True)
dim_location['location_id'] = range(1, len(dim_location) + 1)
dim_location = dim_location[['location_id', 'city_type']]

print(f"dim_location: {dim_location.shape[0]} unique location types")
print(dim_location)

dim_location: 3 unique location types
   location_id     city_type
0            1         Urban
1            2  Metropolitan
2            3    Semi-Urban


In [6]:
# ==============================================================
# DIM_CONDITIONS — one row per weather + traffic combination
# ==============================================================
# Combining weather + traffic into one dim lets us analyze 
# "sunny + jam" vs "sunny + low" cleanly.

dim_conditions = df[['weather', 'traffic']].drop_duplicates().reset_index(drop=True)
dim_conditions['condition_id'] = range(1, len(dim_conditions) + 1)
dim_conditions = dim_conditions[['condition_id', 'weather', 'traffic']]

print(f"dim_conditions: {dim_conditions.shape[0]} unique weather+traffic combos")
print(dim_conditions)

dim_conditions: 24 unique weather+traffic combos
    condition_id     weather traffic
0              1       Sunny    High
1              2      Stormy     Jam
2              3  Sandstorms     Low
3              4       Sunny  Medium
4              5      Cloudy    High
5              6      Cloudy     Jam
6              7         Fog     Jam
7              8      Cloudy  Medium
8              9      Stormy    High
9             10  Sandstorms  Medium
10            11  Sandstorms     Jam
11            12       Windy    High
12            13       Windy     Jam
13            14      Cloudy     Low
14            15       Windy     Low
15            16      Stormy     Low
16            17       Sunny     Low
17            18         Fog     Low
18            19       Sunny     Jam
19            20       Windy  Medium
20            21         Fog  Medium
21            22      Stormy  Medium
22            23  Sandstorms    High
23            24         Fog    High


In [7]:
# ==============================================================
# FACT_ORDERS — the center of the star schema
# ==============================================================
# One row per delivery, with foreign keys to all dimensions,
# plus the numeric measures (delivery_minutes, distance_km, is_on_time).

# Start by merging dimension IDs back into the main df
# Merge restaurant_id
df = df.merge(dim_restaurant[['restaurant_id', 'restaurant_key']], 
              on='restaurant_key', how='left')

# Merge date_id
df = df.merge(dim_date[['date_id', 'order_date']], on='order_date', how='left')

# Merge location_id
df = df.merge(dim_location, on='city_type', how='left')

# Merge condition_id
df = df.merge(dim_conditions, on=['weather', 'traffic'], how='left')

# Build fact_orders with only the foreign keys and measures
fact_orders = df[[
    'order_id',
    'courier_id',       # FK to dim_courier
    'restaurant_id',    # FK to dim_restaurant
    'date_id',          # FK to dim_date
    'location_id',      # FK to dim_location
    'condition_id',     # FK to dim_conditions
    # Attributes specific to this order (not reused across orders)
    'order_type',
    'vehicle_type',
    'multi_deliveries',
    'is_festival',
    # Measures — the numbers we aggregate
    'delivery_minutes',
    'distance_km',
    'is_on_time',
    'order_hour',
    'is_peak_hour',
    'time_ordered',
    'time_picked',
    'delivery_lat',
    'delivery_lng'
]]

print(f"fact_orders: {fact_orders.shape[0]} rows, {fact_orders.shape[1]} columns")
print(fact_orders.head())

# Sanity check — no nulls in foreign keys (would indicate broken joins)
print("\nNulls in FK columns (should all be 0):")
print(fact_orders[['courier_id', 'restaurant_id', 'date_id', 
                    'location_id', 'condition_id']].isnull().sum())

fact_orders: 39141 rows, 19 columns
  order_id       courier_id  restaurant_id  date_id  location_id  \
0   0x4607   INDORES13DEL02              1        1            1   
1   0xb379   BANGRES18DEL02              2        2            2   
2   0x5d6d   BANGRES19DEL01              3        1            1   
3   0x7a6a  COIMBRES13DEL02              4        3            2   
4   0x70a2   CHENRES12DEL01              5        4            2   

   condition_id order_type vehicle_type  multi_deliveries is_festival  \
0             1      Snack   motorcycle                 0          No   
1             2      Snack      scooter                 1          No   
2             3     Drinks   motorcycle                 1          No   
3             4     Buffet   motorcycle                 1          No   
4             5      Snack      scooter                 1          No   

   delivery_minutes  distance_km  is_on_time  order_hour  is_peak_hour  \
0                24         3.03          

In [8]:
# ==============================================================
# SAVE ALL TABLES
# ==============================================================
# Next step (Day 5) will load these into DuckDB.

fact_orders.to_csv("../data/processed/fact_orders.csv", index=False)
dim_courier.to_csv("../data/processed/dim_courier.csv", index=False)
dim_restaurant.to_csv("../data/processed/dim_restaurant.csv", index=False)
dim_date.to_csv("../data/processed/dim_date.csv", index=False)
dim_location.to_csv("../data/processed/dim_location.csv", index=False)
dim_conditions.to_csv("../data/processed/dim_conditions.csv", index=False)

print("All tables saved to data/processed/")
print(f"\nSummary:")
print(f"  fact_orders:    {len(fact_orders):>6} rows")
print(f"  dim_courier:    {len(dim_courier):>6} rows")
print(f"  dim_restaurant: {len(dim_restaurant):>6} rows")
print(f"  dim_date:       {len(dim_date):>6} rows")
print(f"  dim_location:   {len(dim_location):>6} rows")
print(f"  dim_conditions: {len(dim_conditions):>6} rows")

All tables saved to data/processed/

Summary:
  fact_orders:     39141 rows
  dim_courier:      1170 rows
  dim_restaurant:    388 rows
  dim_date:           44 rows
  dim_location:        3 rows
  dim_conditions:     24 rows
